# Lab 04 (solution): Memory and context

Reference implementation. The two disciplines that keep a long-running agent coherent: state vs. memory with checkpoint/rollback, and context-budget assembly under a token budget. Concepts: [state-vs-memory](../../../concepts/memory/state-vs-memory.md), [context-engineering](../../../concepts/context/context-engineering.md).

## Step 0: state vs. memory, with checkpoints

In [ ]:
import sys
sys.path.insert(0, "..")  # modules (checkpoints.py, context_budget.py) live in the lab root
from checkpoints import run_scenario
# State is the current task (rewindable); memory is durable (carries across tasks). Snapshot state
# before each step so a correction is cheap.
base, _ = run_scenario()
print("initial plan (budget 200/day):", base.spends())

## Step 1: a mid-task correction via rollback

In [ ]:
# The user changes their mind after day 2: raise the budget. Roll back to before day 3 - keep days
# 1-2, re-plan only 3-5 - while the new budget (memory) survives the rollback.
p, calls = run_scenario(correction_budget=350)
print("corrected plan:", p.spends())
print(f"steps recomputed: {p.compute_calls - calls} (only days 3-5); memory budget now {p.memory['budget_per_day']}")

## Step 2: the context window is a budget

In [ ]:
from context_budget import assemble, INSTRUCTIONS, MEMORY, HISTORY
# The context window is a budget shared by instructions, memory, and history. Under pressure you must
# SELECT what matters and COMPRESS the rest - a bigger window is not a substitute for selection.
tight = assemble(INSTRUCTIONS, MEMORY, HISTORY, budget=40)
for c in tight["context"]:
    print(f"  [{c['kind']:>12}] {c['text']}")
print("dropped:", [d["text"][:30] for d in tight["dropped"]])

## Step 3: same logic, looser budget (write / select / compress / isolate)

In [ ]:
# Same inputs, a generous budget: nothing is dropped. The assembly logic is identical; only the
# pressure changed. Real systems count tokens (here we approximate by words) and add WRITE (keep
# durable facts in external memory) and ISOLATE (give sub-tasks their own context).
loose = assemble(INSTRUCTIONS, MEMORY, HISTORY, budget=1000)
print("loose budget used:", loose["used"], " dropped:", len(loose["dropped"]))

## What you built

The two disciplines that keep a long-running agent coherent. **State vs. memory with checkpoints**: state is the rewindable current task and memory is the durable carry-across, so a mid-task correction rolls back only the affected steps while preferences persist - react fast with state, learn slowly with memory. **Context-budget assembly**: when instructions, memory, and history compete for a finite window, you SELECT the highest-priority items and COMPRESS the overflow, because models attend unevenly and degrade as context grows (context rot) - a bigger window does not remove the need to choose.

**Where this simplifies:** token cost is approximated by word count (real systems count tokens), priority is a fixed relevance/recency score (real systems learn or compute it), and compression is a placeholder note (real compaction summarizes with a model, keeping pointers to content recoverable from the environment). The moves - write, select, compress, isolate - are the production ones. Concepts: [state-vs-memory](../../../concepts/memory/state-vs-memory.md), [context-engineering](../../../concepts/context/context-engineering.md), [context-rot-and-failure-modes](../../../concepts/context/context-rot-and-failure-modes.md).